# Базовый случай: BM25

## Цель

Построим и оценим базовый подход к кандидатогенерации – лексический поиск через BM25 по предобработанному тексту объявлений (заголовок + описание + параметры, лемматизированные). В результате получим `experiments.csv` с оценкой метода по метрике recall@50. Это будет точка отсчёта для сравнения с последующими подходами.

## Подготовка

### Импорты

In [2]:
import pandas as pd

from avito_candidate_gen.data import (
    load_train,
    train_val_split,
    DATA_PROCESSED_DIR,
    EXPERIMENTS_DIR,
)
from avito_candidate_gen.preprocessing import (
    get_or_build_processed_items,
    build_query_text,
)
from avito_candidate_gen.retrieval.bm25 import BM25Retriever
from avito_candidate_gen.metrics import recall_at_50

### Предобработка данных

Предобработка (лемматизация, сборка текстового поля) считается один раз и кэшируется на диск (`get_or_build_processed_items`). При повторном запуске ноутбука позволяет считывать готовый файл, а не пересчитывается заново.

In [3]:
df_train = load_train()
df_train_processed = get_or_build_processed_items(
    df_train, DATA_PROCESSED_DIR / "train_processed.parquet"
)

Читаю кэш: /Users/xxxogan/study/avito_test/avito-candidate-gen/data/processed/train_processed.parquet


In [4]:
df_train_processed.head(3)

,search_query,search_location_id,search_is_delivery_search,search_infm_params_text,search_category,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_infm_params_text,item_id,item_description_raw,item_category_id,item_text_processed
0,скупка телевизоров,652430,0,,114,Скупка б/у техники,91.0,4.989011,1.0,2303374,39.712818,652430,54.629230,True,False,Вид услуги Место оказания услуг Первомайский п...,e8b685dffe1a408e,Скупаю практически любую современную новую и б...,114,скупка б у техника скупка б у техника скупать ...
1,автоподбор,640860,0,Рейтинг пользователя 4 звезды и выше,114,Автоподбор Разовый осмотр автомобиля,462.0,4.982684,3500.0,2303374,44.051010,640860,56.273390,False,False,Вид услуги Место оказания услуг Нижний Новгоро...,92e1b0446f827b59,🚗 Автоподбор и выездная диагностика автомобиля...,114,автоподбор разовый осмотр автомобиль автоподбо...
2,баня на дровах,653240,0,"Онлайн-запись Тип услуги СПА-услуги, массаж Ви...",114,"Баня на дровах ""Прованс"" на Цветочной",NaN,NaN,1600.0,86469,30.128784,653240,59.783688,False,False,"Вид услуги Красота, здоровье Место оказания ус...",624846856ce81d69,В ритме современной жизни так сложно найти мо...,114,баня на дрова прованс на цветочный баня на дро...


### Сплит и корпус

In [5]:
df_train_split, df_val_split = train_val_split(df_train_processed)

# Корпус для поиска на этапе валидации
corpus_items = df_train_processed.drop_duplicates(subset="item_id")[
    ["item_id", "item_text_processed"]
]

print(f"Train: {len(df_train_split):,} строк")
print(f"Val: {len(df_val_split):,} строк")
print(f"Корпус для поиска: {len(corpus_items):,} уникальных объявлений")

Train: 403,091 строк
Val: 94,582 строк
Корпус для поиска: 344,825 уникальных объявлений


Запросы, по которым будет считаться recall@50, берутся только из val-части, train-запросы в оценке метрики не участвуют.

## Построение BM25-индекса

In [6]:
retriever = BM25Retriever.from_dataframe(corpus_items)

Split strings:   0%|          | 0/344825 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/344825 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/344825 [00:00<?, ?it/s]

Индекс постоен успешно

## Подготовка val-запросов

In [7]:
val_queries = df_val_split.groupby("search_query").first().reset_index()
val_queries["query_text"] = val_queries["search_query"].apply(build_query_text)

relevant: dict[str, set[str]] = {
    str(query): items
    for query, items in df_val_split.groupby("search_query")["item_id"]
    .apply(set)
    .to_dict()
    .items()
}
print(f"Уникальных val-запросов: {len(val_queries):,}")
val_queries[["search_query", "query_text"]].head()

Уникальных val-запросов: 14,906


,search_query,query_text
0,"""промывка радиатора автомобиля без снятия любе...",промывка радиатор автомобиль без снятие люберцы
1,"""ремонт ванной под ключ""",ремонт ванная под ключ
2,'ktrnhbrf ghjdjlrf,ktrnhbrf ghjdjlrf
3,0445110298,0445110298
4,1,1


Среди примеров встречаются «мусорные» запросы:
- опечатки раскладки клавиатуры, например, `ktrnhbrf ghjdjlrf`;
- какой-то артикул (`0445110298`);
- односимвольные запросы — `1`.

Для таких случаев лексический поиск в принципе не может дать релевантный результат. Это ожидаемая особенность реальных пользовательских данных, а не ошибка. Масштаб влияния на recall можно будет оценить после замера метрики.

## Поиск и оценка

Запускаем батчевый поиск BM25 по всем val-запросам и формируем словарь predictions: `search_query` -> топ-50 `item_id`.

In [8]:
top50_lists = retriever.search_batch(val_queries["query_text"].tolist(), top_k=50)
predictions = dict(zip(val_queries["search_query"], top50_lists))

Split strings:   0%|          | 0/14906 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/14906 [00:00<?, ?it/s]

Считаем целевую метрику Recall@50, на val-запросах

In [9]:
score = recall_at_50(predictions, relevant)
print(f"BM25 baseline Recall@50: {score:.4f}")

BM25 baseline Recall@50: 0.3534


## Запись результата

In [10]:
experiment_row = pd.DataFrame([{
    "approach": "bm25_baseline",
    "params": "title_repeats=2, lemmatized (pymorphy3), corpus=344825 unique train items",
    "recall_at_50": score,
    "comment": "title+description+infm_params combined text"
}])

exp_path = EXPERIMENTS_DIR / "experiments.csv"
experiment_row.to_csv(exp_path, mode="a", header=not exp_path.exists(), index=False)
print(f"Записано в {exp_path}")

Записано в /Users/xxxogan/study/avito_test/avito-candidate-gen/experiments/experiments.csv


## Проверка результата

Прежде чем двигаться дальше, проверим конкретные запросы вручную, чтобы убедиться, что низкое значение recall отражает реальную сложность задачи, а не техническую ошибку в пайплайне

In [12]:
sample_query = val_queries.iloc[0]["search_query"]
print("Запрос:", sample_query)
print("Предсказано (первые 10):", predictions[sample_query][:10])
print("Релевантно:", relevant[sample_query])

Запрос: "промывка радиатора автомобиля без снятия люберцы"
Предсказано (первые 10): ['4965e6b5c378ba7b', '97b19fef19b7f1cc', 'aae3c568bf883ef9', '4c0c82d777656e4b', 'a28375592b0b8086', '984907f2c811cc34', 'a0c07009a4e306a7', 'e26691430ba17f46', '4d2490f1e9d9b595', '4ffa074092112717']
Релевантно: {'791cc82871ec82d7'}


In [13]:
# Что было в релевантном объявлении
relevant_id = list(relevant[sample_query])[0]
print("Релевантное объявление:")
print(corpus_items[corpus_items["item_id"] == relevant_id][["item_id", "item_text_processed"]])

print("\nЧто предсказал BM25 (топ-3):")
print(corpus_items[corpus_items["item_id"].isin(predictions[sample_query][:3])][["item_id", "item_text_processed"]])

print("\nТекст самого запроса после обработки:")
print(val_queries[val_queries["search_query"] == sample_query]["query_text"].values)

Релевантное объявление:
                item_id                                item_text_processed
61066  791cc82871ec82d7  промывка радиатор промывка радиатор промывка р...

Что предсказал BM25 (топ-3):
                 item_id                                item_text_processed
79342   4965e6b5c378ba7b  промывка радиатор печка без снятие промывка ра...
239201  97b19fef19b7f1cc  промывка радиатор печка отопитель без снятие п...
314342  aae3c568bf883ef9  промывка радиатор грузовой и легковой промывка...

Текст самого запроса после обработки:
<ArrowStringArray>
['промывка радиатор автомобиль без снятие люберцы']
Length: 1, dtype: str


BM25 отработал корректно, предсказания тематически соответствуют сути запроса. Промахи объяснимы и являются свойством задачи: релевантное объявление часто имеет более общий текст, тогда как BM25 объективно предпочитает более точные лексические совпадения. Формат `item_id` и сопоставление предсказание/релевантность работают верно.

Отдельное наблюдение: в тексте запроса встречается название локации («люберцы»), склеенное с текстом услуги без разделителя. Такие токены уводят BM25 на объявления, где локация упомянута в тексте, вместо объявлений с нужной географией. Масштаб паттерна стоит измерить отдельно — локацию правильнее передавать отдельным бустом по `item_location_id`, а не через текстовый поиск.